**Q1 → Investigation queue**

**Q7 → Provider portfolio risk (joined tables)**

**Q9 → Specialty fraud concentration****

That tells a complete story:

**-claim level** (cell no 3)

**-provider level** (cell no 10)

**-portfolio level**(cell no 14)

## **1.FRAUD TEAM QUERIES (Claim Investigation)**

In [0]:
%sql
--Need to execute
SELECT claim_id,
       provider_name,
       claim_amount,
       fraud_score,
       fraud_trigger_source,
       fraud_severity
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE fraud_flag = 'YES'
ORDER BY fraud_score DESC;

--Shows immediate investigation queue ranked by fraud intensity.

In [0]:
%sql
SELECT claim_id,
       provider_name,
       duplicate_claim_reason,
       claim_value_reason,
       payment_integrity_reason,
       billing_pattern_reason
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE fraud_flag IN ('YES','REVIEW');

--Fraud analysts can see exact rule explanations, not just flags.

In [0]:
%sql
SELECT claim_id,
       provider_name,
       provider_risk_level,
       fraud_score
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE fraud_trigger_source = 'PROVIDER_RISK'
ORDER BY fraud_score DESC;

## **2.FINANCE TEAM QUERIES**

In [0]:
%sql
--Total financial exposure of suspicious claims
SELECT
    SUM(claim_amount) AS suspicious_billed_amount,
    SUM(paid_amount) AS suspicious_paid_amount
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE fraud_flag IN ('YES','REVIEW');
--This is the insurer’s potential financial exposure to fraud.

In [0]:
%sql
SELECT claim_id,
       provider_name,
       claim_amount,
       paid_amount,
       payment_integrity_status
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE payment_integrity_status = 'TRIGGERED'
ORDER BY paid_amount - claim_amount DESC;

--Finance teams monitor leakage where payments exceed expected values.

## **3.PROVIDER RISK MANAGEMENT QUERIES**

In [0]:
%sql
SELECT
    g.provider_name,
    COUNT(*) AS suspicious_claims,
    SUM(g.claim_amount) AS suspicious_amount
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals g
WHERE g.fraud_flag IN ('YES','REVIEW')
GROUP BY g.provider_name
ORDER BY suspicious_claims DESC;

--Helps identify providers needing audit or contract review.

In [0]:
%sql
SELECT
    s.provider_id,
    s.name,
    s.specialty,
    s.total_claims,
    s.total_amount,
    COUNT(g.claim_id) AS suspicious_claims,
    SUM(g.claim_amount) AS suspicious_amount
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_claim_summary s
LEFT JOIN dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals g
    ON s.provider_id = g.provider_id
    AND g.fraud_flag IN ('YES','REVIEW')
GROUP BY
    s.provider_id,
    s.name,
    s.specialty,
    s.total_claims,
    s.total_amount
ORDER BY suspicious_amount DESC;

--This shows provider portfolio risk by combining billing volume with fraud signals.

## **4.OPERATIONS TEAM QUERIES**

In [0]:
%sql
SELECT claim_id,
       provider_name,
       claim_amount,
       fraud_score,
       fraud_trigger_source
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE fraud_flag = 'REVIEW'
ORDER BY fraud_score DESC;

--Operations teams use this as a work queue for claim examiners.

In [0]:
%sql
SELECT specialty,
       COUNT(*) AS fraud_cases,
       SUM(claim_amount) AS fraud_amount
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals
WHERE fraud_flag = 'YES'
GROUP BY specialty
ORDER BY fraud_cases DESC;

--Helps leadership understand which treatment areas need tighter controls.

## **5.EXECUTIVE DASHBOARD QUERY**

In [0]:
%sql
SELECT
    s.name AS provider_name,
    s.total_claims,
    COUNT(g.claim_id) AS suspicious_claims,
    ROUND(COUNT(g.claim_id) * 100.0 / s.total_claims,2) AS fraud_rate_pct
FROM dlt_hc_fraud_detection.bronze_silver_gold.gold_claim_summary s
LEFT JOIN dlt_hc_fraud_detection.bronze_silver_gold.gold_fraud_signals g
    ON s.provider_id = g.provider_id
    AND g.fraud_flag IN ('YES','REVIEW')
GROUP BY s.name, s.total_claims
ORDER BY fraud_rate_pct DESC;

--This gives leadership a risk ranking of providers based on fraud signal density.